In [19]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from scipy.optimize import curve_fit, OptimizeWarning
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.simplefilter("ignore", OptimizeWarning)

# Paths for Kaggle or Local Environment
# Mount the MRI NIfTI directory and the Clinical Data spreadsheet accordingly
DATA_DIR = "/kaggle/input/datasets/stacyvangepuram/mu-glioma-post/PKG - MU-Glioma-Post/MU-Glioma-Post"
CLINICAL_XLSX = "/kaggle/input/datasets/stacyvangepuram/mu-glioma-post-clinicaldata/MU-Glioma-Post_ClinicalData-July2025.xlsx"
OUTPUT_DIR = "/kaggle/working/results/progression"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

In [20]:
def sitk_from_nib(img_nii):
    """Convert nibabel NIfTI to SimpleITK image."""
    data = img_nii.get_fdata().astype(np.float32)
    itk_img = sitk.GetImageFromArray(data)
    itk_img.SetSpacing(img_nii.header.get_zooms()[:3])
    itk_img.SetOrigin(img_nii.affine[:3, 3])
    return itk_img


def nib_from_sitk(itk_img):
    """Convert SimpleITK image back to nibabel NIfTI (array + affine)."""
    data = sitk.GetArrayFromImage(itk_img)
    spacing = itk_img.GetSpacing()
    affine = np.eye(4, dtype=np.float32)
    affine[0, 0] = spacing[2]
    affine[1, 1] = spacing[1]
    affine[2, 2] = spacing[0]
    affine[:3, 3] = itk_img.GetOrigin()
    return data, affine


def simple_skull_strip(itk_img, closing_radius=3):
    """
    Very simple skull stripping using intensity threshold + morphology.
    """
    otsu_mask = sitk.OtsuThreshold(itk_img, 0, 1, 200)

    closing = sitk.BinaryMorphologicalClosingImageFilter()
    closing.SetKernelRadius(closing_radius)
    brain_mask = closing.Execute(otsu_mask)

    brain = sitk.Mask(itk_img, brain_mask)
    return brain, brain_mask


def n4_bias_correction(itk_img, mask=None):
    """
    N4 bias field correction using SimpleITK (basic usage).
    """
    if mask is None:
        mask = sitk.OtsuThreshold(itk_img, 0, 1, 200)

    img_float = sitk.Cast(itk_img, sitk.sitkFloat32)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrected = corrector.Execute(img_float, mask)
    return corrected


def register_to_reference(moving_img, fixed_img):
    """
    Rigid registration of moving_img to fixed_img using SimpleITK.
    """
    initial_transform = sitk.CenteredTransformInitializer(
        fixed_img,
        moving_img,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY,
    )

    registration_method = sitk.ImageRegistrationMethod()
    registration_method.SetMetricAsMattesMutualInformation(numberOfHistogramBins=32)
    registration_method.SetMetricSamplingStrategy(registration_method.RANDOM)
    registration_method.SetMetricSamplingPercentage(0.2)
    registration_method.SetInterpolator(sitk.sitkLinear)

    registration_method.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0,
        minStep=1e-4,
        numberOfIterations=100,
        gradientMagnitudeTolerance=1e-6,
    )
    registration_method.SetOptimizerScalesFromPhysicalShift()

    registration_method.SetInitialTransform(initial_transform, inPlace=False)
    registration_method.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
    registration_method.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
    registration_method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    final_transform = registration_method.Execute(fixed_img, moving_img)

    moving_resampled = sitk.Resample(
        moving_img,
        fixed_img,
        final_transform,
        sitk.sitkLinear,
        0.0,
        moving_img.GetPixelID(),
    )

    return moving_resampled, final_transform


def preprocess_longitudinal_scan(scan_path, reference_img=None, target_spacing=(1.0, 1.0, 1.0)):
    """
    Preprocess individual MRI scan for longitudinal analysis.
    """
    itk_img = sitk.ReadImage(str(scan_path))

    # Skull stripping
    brain_img, brain_mask = simple_skull_strip(itk_img)

    # N4 bias correction
    corrected = n4_bias_correction(brain_img, mask=brain_mask)

    # Optional registration to baseline
    if reference_img is not None:
        corrected, _ = register_to_reference(corrected, reference_img)

    # Intensity normalization (z-score inside brain mask)
    corrected_arr = sitk.GetArrayFromImage(corrected).astype(np.float32)
    mask_arr = sitk.GetArrayFromImage(brain_mask).astype(bool)

    if mask_arr.sum() > 0:
        brain_vals = corrected_arr[mask_arr]
        if np.std(brain_vals) > 0:
            corrected_arr = (corrected_arr - brain_vals.mean()) / brain_vals.std()

    corrected_itk = sitk.GetImageFromArray(corrected_arr)
    corrected_itk.CopyInformation(corrected)

    # Resample to target spacing
    orig_spacing = corrected_itk.GetSpacing()
    orig_size = corrected_itk.GetSize()

    new_spacing = target_spacing
    new_size = [
        int(round(osz * (ospc / nspc)))
        for osz, ospc, nspc in zip(orig_size, orig_spacing, new_spacing)
    ]

    resampled = sitk.Resample(
        corrected_itk,
        new_size,
        sitk.Transform(),
        sitk.sitkLinear,
        corrected_itk.GetOrigin(),
        new_spacing,
        corrected_itk.GetDirection(),
        0.0,
        corrected_itk.GetPixelID(),
    )

    data_resampled = sitk.GetArrayFromImage(resampled).astype(np.float32)

    return data_resampled, new_spacing

In [21]:
def extract_tumor_volume(segmentation_path):
    """
    Extract tumor volume from a segmentation mask.
    Returns volume in cm³.
    """
    try:
        seg = nib.load(segmentation_path)
        seg_data = seg.get_fdata()
        tumor_voxels = np.sum(seg_data > 0)
        
        spacing = seg.header.get_zooms()
        voxel_volume_mm3 = spacing[0] * spacing[1] * spacing[2]
        
        return (tumor_voxels * voxel_volume_mm3) / 1000.0
    except Exception as e:
        print(f"Error reading {segmentation_path}: {e}")
        return 0.0

def load_clinical_dates(xlsx_path):
    """
    Parses the clinical spreadsheet to extract the number of days from diagnosis for each MRI.
    Returns a dictionary mapping PatientID -> {timepoint_index: days_from_diagnosis}
    """
    if not os.path.exists(xlsx_path):
        print(f"Warning: Spreadsheet {xlsx_path} not found. Will fallback to simulated intervals.")
        return None
        
    try:
        df = pd.read_excel(xlsx_path)
        patient_dates = {}
        
        # Typically the column is named 'PatientID' or 'Subject'
        id_col = 'PatientID' if 'PatientID' in df.columns else df.columns[0]
        
        # Find all columns indicating Timepoint Days
        day_cols = [c for c in df.columns if "Number of Days from Diagnosis to" in str(c) and "MRI" in str(c)]
        
        for _, row in df.iterrows():
            pid = str(row[id_col]).strip()
            t_dict = {}
            for col in day_cols:
                # Naive extraction of timepoint number, e.g., "... 1st MRI (Timepoint_1)" -> 1
                try:
                    tp_str = col.split('Timepoint_')[1].replace(')', '')
                    tp_idx = int(tp_str) - 1 # 0-indexed internally
                    val = row[col]
                    if pd.notna(val):
                        t_dict[tp_idx] = float(val)
                except:
                    pass
            patient_dates[pid] = t_dict
            
        return patient_dates
    except Exception as e:
        print(f"Error parsing clinical dates: {e}")
        return None

def extract_all_volumes(data_dir, xlsx_path):
    """
    Scans the patient tree and links tumor volumes to real elapsed days.
    Uses *tumorMask*.nii files for volume computation.
    """
    clinical_dates = load_clinical_dates(xlsx_path)
    records = []
    
    if not os.path.exists(data_dir):
        print(f"Warning: {data_dir} does not exist.")
        return pd.DataFrame()
        
    patients = [d for d in sorted(os.listdir(data_dir)) if os.path.isdir(os.path.join(data_dir, d))]
    
    for patient_id in patients:
        patient_path = os.path.join(data_dir, patient_id)
        timepoints = sorted(d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d)))
        
        baseline_days = None
        patient_date_map = clinical_dates.get(patient_id, {}) if clinical_dates else {}
        
        for i, tp in enumerate(timepoints):
            tp_path = os.path.join(patient_path, tp)
            
            # Prefer tumor mask files
            mask_candidates = [
                f for f in os.listdir(tp_path)
                if "tumorMask" in f and (f.endswith(".nii") or f.endswith(".nii.gz"))
            ]
            
            if not mask_candidates:
                # No tumor mask at this timepoint; skip
                # print(f"No tumorMask file for {patient_id} {tp}, skipping.")
                continue
            
            seg_path = os.path.join(tp_path, sorted(mask_candidates)[0])
            volume = extract_tumor_volume(seg_path)
            
            # Determine days from baseline
            absolute_days = patient_date_map.get(i)
            
            if absolute_days is not None:
                if baseline_days is None:
                    baseline_days = absolute_days
                days_from_baseline = absolute_days - baseline_days
            else:
                # Fallback if no data specific for this timepoint
                days_from_baseline = i * 90.0
                
            records.append({
                "patient_id": patient_id,
                "timepoint": tp,
                "tp_index": i,
                "days": days_from_baseline,
                "volume_cm3": volume,
            })

    return pd.DataFrame(records)

# Extract dataset (skipped locally if paths aren't initialized)
print("Scanning dataset to extract features...")
volume_df = extract_all_volumes(DATA_DIR, CLINICAL_XLSX)
if len(volume_df) > 0:
    print(f"✅ Extracted {len(volume_df)} total records across {len(volume_df['patient_id'].unique())} patients.")
else:
    print("⚠️ Dataset not found. (Expected since we aren't executing directly on Kaggle nodes yet).")

Scanning dataset to extract features...
✅ Extracted 594 total records across 203 patients.


In [22]:
class TumorGrowthModels:
    @staticmethod
    def exponential(t, V0, k):
        return V0 * np.exp(k * t)

    @staticmethod
    def gompertz(t, V0, Vmax, k):
        return Vmax * np.exp(-np.log(Vmax / V0) * np.exp(-k * t))

    @staticmethod
    def logistic(t, V0, Vmax, k):
        return Vmax / (1 + ((Vmax / V0) - 1) * np.exp(-k * t))

    @staticmethod
    def linear(t, V0, k):
        return V0 + k * t

In [23]:
def calculate_aic(n, rss, k):
    """Calculates Akaike Information Criterion for model selection."""
    if n <= 0 or rss <= 0: return np.inf
    return 2 * k + n * np.log(rss / n)

def fit_model(times, volumes, model_func, p0=None, bounds=None):
    """Core fitting function providing parameters, R2, AIC, and covariance structures."""
    try:
        if bounds:
            params, pcov = curve_fit(model_func, times, volumes, p0=p0, bounds=bounds, maxfev=10000)
        else:
            params, pcov = curve_fit(model_func, times, volumes, p0=p0, maxfev=10000)

        preds = model_func(times, *params)
        rss = np.sum((volumes - preds)**2)
        aic = calculate_aic(len(times), rss, len(params))
        r2 = r2_score(volumes, preds) if len(volumes) > 1 else 0.0

        return {
            "params": params,
            "pcov": pcov,
            "aic": aic,
            "r2": r2,
            "mae": mean_absolute_error(volumes, preds),
            "predictions": preds,
        }
    except Exception:
        return None

def fit_all_patient_models(times, volumes):
    """Applies all suitable models sequentially based on timepoint length constraints."""
    n = len(times)
    V0 = volumes[0]
    Vmax_obs = max(volumes)
    Vmax_guess = Vmax_obs * 2.0

    results = {}

    if n >= 2:
        slope = (volumes[-1] - volumes[0]) / max(1e-3, (times[-1] - times[0]))
        results["linear"] = fit_model(
            times, volumes, TumorGrowthModels.linear,
            p0=[V0, slope], bounds=([0, -1e3], [1e6, 1e3])
        )
        results["exponential"] = fit_model(
            times, volumes, TumorGrowthModels.exponential,
            p0=[V0, 0.002], # Better initial guess for growth rate
             bounds=([1e-3, -0.01], [1e5, 0.01])
        )# Tighter realistic growth limits
   

    # 3-parameter models require at 4 points to avoid total underdetermination
    if n >= 4:
        results["gompertz"] = fit_model(
            times, volumes, TumorGrowthModels.gompertz,
            p0=[V0, Vmax_guess, 0.001],
            bounds=([1e-3, Vmax_obs, -0.05], [1e5, 15 * Vmax_obs, 0.05])
        )
        results["logistic"] = fit_model(
            times, volumes, TumorGrowthModels.logistic,
            p0=[V0, Vmax_guess, 0.001],
            bounds=([1e-3, Vmax_obs, -0.01], [1e5, 5 * Vmax_obs, 0.01])
        )

    # Strip failed fits
    return {k: v for k, v in results.items() if v is not None and np.isfinite(v["aic"])}

def select_best_model(model_results, n_points=None):
     if not model_results:
         return None

    # Prefer biologically bounded models
     if n_points is not None and n_points >= 4:
         preferred = ["logistic", "gompertz"]
         available = [
            m for m in preferred
            if m in model_results
        ]
         if available:
             return min(available, key=lambda k: model_results[k]["aic"])

     return min(model_results.keys(), key=lambda k: model_results[k]["aic"])

In [24]:
def calculate_vde(v1, v2, delta_days):
    """Velocity of Diametric Expansion in mm/month."""
    if delta_days <= 0: return 0.0
    d1 = (6 * v1 / np.pi)**(1/3) * 10
    d2 = (6 * v2 / np.pi)**(1/3) * 10
    return (d2 - d1) / (delta_days / 30.437)

def get_confidence_bands(model_func, future_times, params, pcov, n_samples=200):
    """Generates 90% confidence bands from parameter covariance (bootstrap sampling)."""
    try:
        samples = np.random.multivariate_normal(params, pcov, n_samples)
        preds = np.array([model_func(future_times, *s) for s in samples])
        lower = np.percentile(preds, 5, axis=0)
        upper = np.percentile(preds, 95, axis=0)
        return np.clip(lower, 0, None), np.clip(upper, 0, None)
    except Exception:
        # Fallback for singular matrices
        base = model_func(future_times, *params)
        return base * 0.9, base * 1.1

def temporal_holdout(times, volumes):
    """Trains on N-1 points and predicts the Nth point to calculate Mean Absolute Percentage Error (MAPE)."""
    if len(times) < 3: return None
    
    t_train, v_train = times[:-1], volumes[:-1]
    t_test, v_test = times[-1], volumes[-1]
    
    models = fit_all_patient_models(t_train, v_train)
    best = select_best_model(models)
    
    if not best: return None
    
    func = getattr(TumorGrowthModels, best)
    params = models[best]["params"]
    
    v_pred = func(t_test, *params)
    mae = abs(v_test - v_pred)
    mape = (mae / v_test) * 100 if v_test > 0 else 0
    dir_acc = 100.0 if np.sign(v_pred - v_train[-1]) == np.sign(v_test - v_train[-1]) else 0.0
    
    return {"mape": mape, "mae": mae, "direction_accuracy": dir_acc}

In [25]:
def predict_growth_trajectory(patient_data, horizon_days=180):
    df = patient_data.sort_values("days")
    times = df["days"].values.astype(float)
    volumes = df["volume_cm3"].values.astype(float)
    
    if len(times) < 2:
        return None

    # Smoothing (reduces noisy spikes)
    if len(volumes) >= 3:
        volumes = pd.Series(volumes).rolling(window=3, center=True, min_periods=1).mean().values
    
    # --- Spike Stabilization (NEW) ---
    # Detect sudden unrealistic jumps
    if len(volumes) >= 2:
        recent_ratio = volumes[-1] / max(volumes[-2], 1e-6)
        recent_dt = times[-1] - times[-2]

        # Spike only if large AND fast
        if recent_ratio > 2.5 and recent_dt < 60:
            volumes[-1] = volumes[-2] * 2.0
            
    # Fit models
    models = fit_all_patient_models(times, volumes)
    best_name = select_best_model(models, n_points=len(times))
    if not best_name:
        return None

    model = models[best_name]
    func = getattr(TumorGrowthModels, best_name)

    # Forecast
    last_t = times[-1]
    future_t = np.arange(last_t + 30, last_t + horizon_days + 30, 30)
    future_v = func(future_t, *model["params"])
    last_vol = volumes[-1]

    # Universal Growth Safety
    growth_ratio = future_v[-1] / max(last_vol, 1e-6)
    if growth_ratio > 4:
        future_v = last_vol * (1 + np.linspace(0, 3, len(future_v)))

    # Adaptive Biological Cap
    if len(volumes) >= 2:
        hist_ratio = volumes[-1] / max(volumes[-2], 1e-6)
    else:
        hist_ratio = 1.0

    if hist_ratio > 1.5:
        max_multiplier = 3.0
    elif hist_ratio > 1.2:
        max_multiplier = 2.7
    else:
        max_multiplier = 2.5

    future_v = np.clip(future_v, 0, max_multiplier * last_vol)

    # Confidence Intervals
    ci_lower, ci_upper = get_confidence_bands(
        func, future_t, model["params"], model["pcov"]
    )

    # 6-Month Prediction
    pred_6mo = future_v[5] if len(future_v) >= 6 else future_v[-1]
    pred_6mo = min(pred_6mo, max_multiplier * last_vol)

    abs_growth = pred_6mo - last_vol
    rel_growth = (abs_growth / last_vol) * 100 if last_vol > 0 else 0.0
    vde = calculate_vde(last_vol, pred_6mo, horizon_days)

    # Risk Classification
    if pred_6mo <= 0.2 or rel_growth <= -90:
        rano = "Complete Response (CR)"
        risk = "LOW"
        rec = "Routine follow up. Tumor structurally removed/shrunk."
    elif rel_growth >= 100 or abs_growth >= 20:
        rano = "Progressive Disease (PD)"
        risk = "CRITICAL"
        rec = "URGENT ACTION (7 DAYS) - Neurosurgical consultation immediately."
    elif rel_growth >= 25:
        rano = "Progressive Disease (PD)"
        risk = "HIGH"
        rec = "HIGH PRIORITY (30 DAYS) - Consider treatment modification."
    elif rel_growth <= -50:
        rano = "Partial Response (PR)"
        risk = "LOW"
        rec = "ACTIVE SURVEILLANCE - Continue current effective therapy."
    else:
        rano = "Stable Disease (SD)"
        risk = "MODERATE"
        rec = "ROUTINE MONITORING - Re-evaluate dynamically."

    # Doubling / Half-life
    halflife = None
    doubling = None
    if best_name == "exponential":
        k = model["params"][1]
        if k > 0:
            doubling = np.log(2) / k
        elif k < 0:
            halflife = np.log(0.5) / k

    # Return Output
    return {
        "model_used": best_name,
        "aic": model["aic"],
        "r2": model["r2"],
        "mae": model["mae"],
        "historical_times": times,
        "historical_volumes": volumes,
        "future_times": future_t,
        "future_predictions": future_v,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "current_volume": last_vol,
        "predicted_6mo_volume": pred_6mo,
        "absolute_growth": abs_growth,
        "relative_growth": rel_growth,
        "vde": vde,
        "rano_status": rano,
        "risk_level": risk,
        "recommendation": rec,
        "doubling_days": doubling,
        "halflife_days": halflife,
        "holdout": temporal_holdout(times, volumes),
    }

In [26]:
def visualize_patient(prediction, patient_id, save_path=None):
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"Progression Trajectory: {patient_id}", fontweight="bold", fontsize=18)

    # 1. Growth curve with covariance CI
    ax = axs[0, 0]
    ax.scatter(prediction["historical_times"], prediction["historical_volumes"], c="navy", s=100, label="Observed", zorder=5)
    ax.plot(prediction["future_times"], prediction["future_predictions"], "r-", lw=2, label="Prediction")
    ax.fill_between(prediction["future_times"], prediction["ci_lower"], prediction["ci_upper"], color="red", alpha=0.15, label="90% CI")
    ax.set_title(f"Forecast Model: {prediction['model_used'].capitalize()} (AIC={prediction['aic']:.1f})")
    ax.set_xlabel("Days from Baseline")
    ax.set_ylabel("Volume (cm³)")
    ax.grid(alpha=0.3)
    ax.legend()

    # 2. Instantaneous Growth Rate
    ax = axs[0, 1]
    t_all = np.concatenate([prediction["historical_times"], prediction["future_times"]])
    v_all = np.concatenate([prediction["historical_volumes"], prediction["future_predictions"]])
    rates = np.gradient(v_all, t_all)
    ax.plot(t_all, rates, c="teal", lw=2)
    ax.axhline(0, color="k", ls="--", alpha=0.5)
    ax.axvline(prediction["historical_times"][-1], color="red", ls="--", alpha=0.5, label="Present")
    ax.set_title("Volume Velocity (cm³/day)")
    ax.set_xlabel("Days from Baseline")
    ax.grid(alpha=0.3)

    # 3. Clinical Metrics Bar
    ax = axs[1, 0]
    bars = {"Current Vol": prediction["current_volume"], "6-Mo Vol": prediction["predicted_6mo_volume"], 
            "Abs Δ": prediction["absolute_growth"], "VDE mm/mo": prediction["vde"]}
    bar_plots = ax.bar(bars.keys(), bars.values(), color=["#2c3e50", "#e74c3c", "#3498db", "#9b59b6"])
    for b in bar_plots:
        ax.text(b.get_x() + 0.4, b.get_height(), f"{b.get_height():.1f}", ha="center", va="bottom", fontweight="bold")
    ax.set_title("Quantitative Metrics")
    ax.grid(axis='y', alpha=0.3)

    # 4. Assessment Summary Box
    ax = axs[1, 1]
    ax.axis("off")
    risk_colors = {"CRITICAL": "#c0392b", "HIGH": "#e74c3c", "MODERATE": "#f39c12", "LOW": "#27ae60"}
    
    hold = prediction["holdout"]
    hold_txt = f"Holdout MAPE: {hold['mape']:.1f}%" if hold else "Holdout N/A (<3 points)"
    th_txt = f"Half-Life: {prediction['halflife_days']:.1f}d" if prediction['halflife_days'] else (
             f"Doubling: {prediction['doubling_days']:.1f}d" if prediction['doubling_days'] else "")
    
    text = f"""
CLINICAL ASSESSMENT
{'='*40}
RANO Classification:
{prediction['rano_status']}

Risk Stratification:
{prediction['risk_level']}

Model Validation:
R²: {prediction['r2']:.3f} | {hold_txt}
{th_txt}

Recommendation:
{prediction['recommendation']}
"""
    ax.text(0.5, 0.5, text.strip(), transform=ax.transAxes,
            bbox=dict(boxstyle="round", facecolor=risk_colors.get(prediction["risk_level"], "grey"), alpha=0.15),
            fontsize=12, family="monospace", va="center", ha="center")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.close(fig) # Memory safe

In [27]:
def process_cohort(df):
    if len(df) == 0: return
    pids = df["patient_id"].unique()
    results = []
    
    print(f"Executing forecasting pipeline on {len(pids)} patients...")
    for pid in tqdm(pids, leave=False):
        df_sub = df[df["patient_id"] == pid]
        pred = predict_growth_trajectory(df_sub)
        
        if not pred: continue
        
        # Outlier tracking
        if pred["relative_growth"] > 1000:
            pred["rano_status"] = "INVALID OUTLIER"
            
        visualize_patient(pred, pid, save_path=f"{OUTPUT_DIR}/plots/{pid}_forecast.png")
        
        row = {
            "PatientID": pid,
            "Total_Timepoints": len(df_sub),
            "Baseline_Volume": pred["historical_volumes"][0],
            "Current_Volume": pred["current_volume"],
            "Forecast_6Mo_Volume": pred["predicted_6mo_volume"],
            "VDE_mm_mo": pred["vde"],
            "Best_Model": pred["model_used"],
            "R2": pred["r2"],
            "RANO_Status": pred["rano_status"],
            "Risk_Tier": pred["risk_level"]
        }
        if pred["holdout"]:
            row["Holdout_MAPE"] = pred["holdout"]["mape"]
            row["Holdout_Direction"] = pred["holdout"]["direction_accuracy"]
            
        results.append(row)

    out_df = pd.DataFrame(results)
    out_df.to_csv(f"{OUTPUT_DIR}/progression_forecast_summary.csv", index=False)
    
    print("\n✅ Batch Processing Complete")
    df_2 = out_df[out_df["Total_Timepoints"] == 2]
    df_3 = out_df[out_df["Total_Timepoints"] >= 3]
    
    print("\n--- COHORT EVALUATION REPORT ---")
    print(f"Analyzed {len(out_df)} valid patient trajectories.")
    print(f"Risk Profiling: {out_df['Risk_Tier'].value_counts().to_dict()}")
    
    print(f"\nModel Performance (2 Timepoints, n={len(df_2)}):")
    if len(df_2) > 0: print(f"  Avg R² = {df_2['R2'].mean():.3f}")
    
    print(f"\nModel Performance (3+ Timepoints, n={len(df_3)}):")
    if len(df_3) > 0:
        print(f"  Avg R² = {df_3['R2'].mean():.3f}")
        print(f"  Holdout MAPE = {df_3['Holdout_MAPE'].mean():.1f}%")
        print(f"  Direction Correct = {df_3['Holdout_Direction'].mean():.1f}%")
        
    return out_df

# If running on Kaggle with matching environment, uncomment:
if len(volume_df) > 0:
    summary = process_cohort(volume_df)
    volume_df.to_csv(f"{OUTPUT_DIR}/tumor_volumes_master.csv", index=False)

Executing forecasting pipeline on 203 patients...


  0%|          | 0/203 [00:00<?, ?it/s]


✅ Batch Processing Complete

--- COHORT EVALUATION REPORT ---
Analyzed 150 valid patient trajectories.
Risk Profiling: {'CRITICAL': 56, 'MODERATE': 56, 'LOW': 33, 'HIGH': 5}

Model Performance (2 Timepoints, n=41):
  Avg R² = 0.806

Model Performance (3+ Timepoints, n=109):
  Avg R² = 0.797
  Holdout MAPE = 36.5%
  Direction Correct = 63.8%
